# Drishyam: train the dacl10k segmenter on Kaggle

Generated from `training/train_dacl10k.py` by `training/make_notebook.py` (edit the script, not this notebook).
It trains a UNet + MobileNetV3 on dacl10k, exports the FP32 and FP16 ONNX that Drishyam loads, checks them
against PyTorch on OpenCV 5 DNN, and zips `best.onnx`, `best_fp16.onnx`, `metrics.json` and `run_info.json`.

1. **Data**: *Add Input* > Datasets > search `dacl10k-v2-devphase` and add **winterqi131/dacl10k-v2-devphase**
   (the official v2 devphase files: 6,935 train and 975 validation images with labelme JSON; the identical
   **sristi29raj/dacl10k-dataset** also works). Both are third-party re-uploads; the data's own licence is
   CC BY-NC 4.0 (its `LICENSE.md`). Off Kaggle (Colab, a laptop), `kagglehub.dataset_download(
   "winterqi131/dacl10k-v2-devphase")` fetches the same files without a token; pass its path to `--data`.
   (The official AWS zip answers "All access to this object has been disabled" since 2026-10.)
2. **Settings** (right panel): Accelerator **GPU T4 x2** (one GPU is used), Internet **on** (pip and the
   ImageNet encoder weights; needs a phone-verified account).
3. **Save Version > Save & Run All (Commit)**. It runs in the background (you can close the tab) for up to
   12 h; 30 epochs are estimated at 2 to 4 h on a T4, and `--hours 11` stops early enough to export anyway.
4. When the version is done: its **Output** tab > `run/drishyam_dacl10k.zip` > download.
5. Stopped early ("resume to continue" in the log)? New version: *Add Input* > Your Work > this notebook's
   output, then Save & Run All again; `--resume auto` carries on from its `last.pt`.
6. On your machine, in the repo: `scripts/install_model.sh ~/Downloads/drishyam_dacl10k.zip` (the path the
   browser saved it to; checks, smoke jobs, before/after eval; see training/README.md).

The weights are trained on CC BY-NC 4.0 data, so they are for non-commercial use.

In [ ]:
# Imports
from __future__ import annotations

import argparse
import glob
import json
import math
import os
import random
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [ ]:
# Setup: the repo files and packages the cells below need
# In the notebook make_notebook.py fills REPO_FILES with the training helpers and drishyam/segment.py (for the
# parity check), so it runs on Kaggle without the repo; run from the repo, the files beside this script are used.
REPO_FILES = {  # written by make_notebook.py from the repo: edit those files
    'training/dacl_common.py': '"""Shared by train / export / parity: the class map, the model factory and the 19 -> 6 merge head."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nMEAN = np.array([0.485, 0.456, 0.406], np.float32)  # ImageNet; drishyam.segment.OnnxSegmenter normalises the same way\nSTD = np.array([0.229, 0.224, 0.225], np.float32)\nCLASS_MAP = Path(__file__).with_name("class_map.json")\n\n\ndef load_class_map(path: Path = CLASS_MAP):\n    """(dacl10k names, drishyam names, groups) where groups[c - 1] lists the dacl10k channels merged into\n    Drishyam class c (c = 1..5; background is the constant-0 logit, never a group)."""\n    cm = json.loads(Path(path).read_text())\n    dacl, dri, mapping = cm["dacl10k_classes"], cm["drishyam_classes"], cm["map"]\n    if set(mapping) != set(dacl) or len(set(dacl)) != len(dacl):\n        raise ValueError("class_map.json: \'map\' keys must be exactly the dacl10k classes")\n    if not set(mapping.values()) <= set(dri) or dri[0] != "background":\n        raise ValueError("class_map.json: \'map\' values must be Drishyam classes, index 0 = background")\n    groups = [[i for i, n in enumerate(dacl) if mapping[n] == c] for c in dri[1:]]\n    if not all(groups):\n        raise ValueError("class_map.json: every Drishyam defect class needs at least one dacl10k channel")\n    return dacl, dri, groups\n\n\ndef build_model(arch: str = "unet", encoder: str = "tu-mobilenetv3_large_100",\n                encoder_weights: str | None = "imagenet", classes: int = 19) -> nn.Module:\n    """Plain multi-label segmenter: (B,3,H,W) ImageNet-normalised RGB -> (B,classes,H,W) logits.\n    H and W must be multiples of 32 (encoder stride)."""\n    import segmentation_models_pytorch as smp\n\n    cls = {"unet": smp.Unet, "deeplabv3plus": smp.DeepLabV3Plus}[arch]\n    return cls(encoder_name=encoder, encoder_weights=encoder_weights, classes=classes, activation=None)\n\n\nclass DrishyamHead(nn.Module):\n    """19 sigmoid logits -> 6 softmax logits, inside the graph so the ONNX needs no class_map.\n\n    Channel 0 (background) is the constant logit 0 and defect c is the max of its mapped channels\' logits\n    (= the sigmoid of the max, since sigmoid is monotone). Then softmax([0, s_1..s_5]) gives, for a pixel with\n    a single active class, exactly sigmoid(s_c) for that class and 1 - sigmoid(s_c) for background; when two\n    classes are both active they split the mass in proportion to their odds. A dacl10k class mapped to\n    background simply contributes no evidence. This is what `OnnxSegmenter`\'s softmax expects.\n    """\n\n    def __init__(self, net: nn.Module, groups: list[list[int]]):\n        super().__init__()\n        self.net = net\n        self.n_groups = len(groups)\n        for i, g in enumerate(groups):\n            self.register_buffer(f"g{i}", torch.tensor(g, dtype=torch.long), persistent=False)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = self.net(x)\n        outs = [z[:, :1] * 0.0]  # keeps the dynamic H x W; a constant would need a Shape/Expand chain\n        for i in range(self.n_groups):\n            outs.append(torch.index_select(z, 1, getattr(self, f"g{i}")).amax(dim=1, keepdim=True))\n        return torch.cat(outs, dim=1)\n\n\ndef load_checkpoint(path: str, device="cpu"):\n    """Returns (plain 19-channel net in eval mode, checkpoint dict). Never downloads encoder weights."""\n    ck = torch.load(path, map_location=device)\n    net = build_model(ck["arch"], ck["encoder"], None, ck["classes"])\n    net.load_state_dict(ck["model"])\n    return net.to(device).eval(), ck\n',
    'training/class_map.json': '{\n  "_comment": "dacl10k (19 sigmoid channels) -> Drishyam (6-way) merge. Channel order is dacl10k-toolkit Dacl10kDataset.TARGET_LIST; the model\'s 19 output channels follow it. Edit \'map\' and re-run export_onnx.py: the merge lives in the exported graph, so no retraining is needed.",\n  "source": "https://github.com/phiyodr/dacl10k-toolkit (dacl10k/dacl10kdataset.py TARGET_LIST; README \'Labels/Classes\'); paper Flotzinger et al., WACV 2024, arXiv:2309.00460, appendix Tables 7-9",\n  "drishyam_classes": ["background", "crack", "spalling", "efflorescence", "exposed_rebar", "corrosion"],\n  "dacl10k_classes": [\n    "Crack", "ACrack", "Wetspot", "Efflorescence", "Rust", "Rockpocket", "Hollowareas", "Cavity",\n    "Spalling", "Graffiti", "Weathering", "Restformwork", "ExposedRebars",\n    "Bearing", "EJoint", "Drainage", "PEquipment", "JTape", "WConccor"\n  ],\n  "map": {\n    "Crack": "crack",\n    "ACrack": "crack",\n    "Wetspot": "background",\n    "Efflorescence": "efflorescence",\n    "Rust": "corrosion",\n    "Rockpocket": "spalling",\n    "Hollowareas": "spalling",\n    "Cavity": "spalling",\n    "Spalling": "spalling",\n    "Graffiti": "background",\n    "Weathering": "background",\n    "Restformwork": "background",\n    "ExposedRebars": "exposed_rebar",\n    "Bearing": "background",\n    "EJoint": "background",\n    "Drainage": "background",\n    "PEquipment": "background",\n    "JTape": "background",\n    "WConccor": "spalling"\n  }\n}\n',
    'training/export_onnx.py': '"""Checkpoint -> Drishyam ONNX (FR-4): (1,3,H,W) ImageNet-normalised RGB in, (1,6,H,W) logits out.\n\nThe 19 dacl10k sigmoid channels are merged to Drishyam\'s 6 classes inside the graph (dacl_common.DrishyamHead,\nmapping from class_map.json), so OnnxSegmenter needs no class_map. H and W are dynamic (multiples of 32).\nWrites <out>/drishyam_dacl10k.onnx (FP32) and drishyam_dacl10k.fp16.onnx (weights in FP16, FP32 inputs/outputs).\n\n    python training/export_onnx.py --ckpt runs/dacl10k/best.pt --out models/\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport inspect\nimport sys\nimport warnings\nfrom pathlib import Path\n\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom dacl_common import DrishyamHead, load_checkpoint, load_class_map\n\nNAME = "drishyam_dacl10k"\n\n\ndef export(ckpt: str, out: Path, size: int = 512, slim: bool = True, fp16: bool = True) -> dict[str, Path]:\n    import onnx\n\n    names, _, groups = load_class_map()\n    net, ck = load_checkpoint(ckpt)\n    if ck["names"] != names:\n        raise SystemExit("checkpoint channels do not match class_map.json \'dacl10k_classes\'")\n    model = DrishyamHead(net, groups).eval()\n    out.mkdir(parents=True, exist_ok=True)\n    paths = {"fp32": out / f"{NAME}.onnx"}\n    dummy = torch.randn(1, 3, size, size)\n    # The TorchScript exporter, the one checked against OpenCV DNN here. torch >= 2.9 defaults to dynamo=True\n    # (a different graph, and it needs onnxscript); torch 2.2.2 has no such argument.\n    legacy = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}\n    with torch.no_grad():\n        torch.onnx.export(model, dummy, str(paths["fp32"]), opset_version=17, input_names=["input"],\n                          output_names=["logits"], dynamic_axes={"input": {2: "h", 3: "w"}, "logits": {2: "h", 3: "w"}},\n                          **legacy)\n    if slim:\n        try:\n            import onnxslim\n\n            onnxslim.slim(str(paths["fp32"]), str(paths["fp32"]))  # constant-folds the shape ops OpenCV DNN dislikes\n        except ImportError:\n            print("onnxslim not installed: skipping graph cleanup (pip install onnxslim)")\n    onnx.checker.check_model(onnx.load(str(paths["fp32"])))\n    if fp16:\n        try:\n            from onnxconverter_common import float16\n\n            with warnings.catch_warnings():  # "float32 number 1e-08 will be truncated": harmless underflow of tiny weights\n                warnings.simplefilter("ignore")\n                half = float16.convert_float_to_float16(onnx.load(str(paths["fp32"])), keep_io_types=True)\n            paths["fp16"] = out / f"{NAME}.fp16.onnx"\n            onnx.save(half, str(paths["fp16"]))\n        except ImportError:\n            print("onnxconverter-common not installed: no FP16 variant (pip install onnxconverter-common)")\n    return paths\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    p.add_argument("--ckpt", required=True, help="best.pt / last.pt from train_dacl10k.py")\n    p.add_argument("--out", default="models", help="output directory")\n    p.add_argument("--size", type=int, default=512, help="trace size (the graph stays dynamic)")\n    p.add_argument("--no-slim", action="store_true")\n    p.add_argument("--no-fp16", action="store_true")\n    a = p.parse_args(argv)\n    paths = export(a.ckpt, Path(a.out), a.size, not a.no_slim, not a.no_fp16)\n    for k, v in paths.items():\n        print(f"{k}: {v}  {v.stat().st_size / 1e6:.2f} MB")\n    try:  # the check that matters: does the OpenCV 5 DNN new engine accept the graph, at 2 sizes?\n        import cv2\n        import numpy as np\n\n        for k, v in paths.items():\n            net = cv2.dnn.readNetFromONNX(str(v), engine=cv2.dnn.ENGINE_NEW)\n            for s in (256, a.size):\n                net.setInput(np.zeros((1, 3, s, s), np.float32))\n                print(f"OpenCV {cv2.__version__} DNN ENGINE_NEW {k} @ {s}: out {tuple(np.asarray(net.forward()).shape)}")\n    except ImportError:\n        print("cv2 not installed: skipped the OpenCV DNN load check")\n\n\nif __name__ == "__main__":\n    main()\n',
    'training/parity.py': '"""FR-4 acceptance: the exported ONNX on OpenCV 5 DNN (ENGINE_NEW), through drishyam.segment.OnnxSegmenter,\nmust match the PyTorch checkpoint on 20 fixed crops.\n\n    python training/parity.py --ckpt runs/dacl10k/best.pt --onnx models/drishyam_dacl10k.onnx \\\n        [--onnx-fp16 models/drishyam_dacl10k.fp16.onnx]\n\nCompared per crop: probabilities (softmax over the 6 logits, as OnnxSegmenter returns them) and the raw DNN\nlogits. Crops are 512 x 512 windows of public/samples/*.jpg at fixed positions. Each ONNX runs at 512 (the\n"accurate" path) and at 256 (the "fast" path), which also checks that H x W really is dynamic.\nExit status: 0 when every row is within its tolerance (max |dprob|: FP32 1e-3, FP16 5e-2 by default), 1 when an\nFP32 row is not (do not install), FP16_OVER (3) when FP32 passes and only FP16 is over (install the FP32 file alone:\nthe FP16 margin shrinks as training goes on, so it must not hold back a good FP32 model).\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parent\nsys.path.insert(0, str(HERE))\nsys.path.insert(0, str(ROOT))\nfrom dacl_common import MEAN, STD, DrishyamHead, load_checkpoint, load_class_map\n\nfrom drishyam.segment import OnnxSegmenter\n\nFP16_OVER = 3\n\n\ndef crops(samples: Path, n: int, size: int, seed: int) -> list[np.ndarray]:\n    files = sorted(samples.glob("*.jpg"))\n    if not files:\n        raise SystemExit(f"no *.jpg in {samples}")\n    imgs = [cv2.imread(str(f)) for f in files]\n    rng = np.random.default_rng(seed)\n    out = []\n    for i in range(n):\n        img = imgs[i % len(imgs)]\n        h, w = img.shape[:2]\n        y, x = int(rng.integers(0, h - size + 1)), int(rng.integers(0, w - size + 1))\n        out.append(img[y:y + size, x:x + size].copy())\n    return out\n\n\ndef torch_reference(model, crop: np.ndarray, size: int):\n    """(6, size, size) logits and the (6, h, w) probabilities, preprocessed like OnnxSegmenter.predict."""\n    h, w = crop.shape[:2]\n    rgb = cv2.cvtColor(cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)\n    blob = ((rgb.astype(np.float32).transpose(2, 0, 1) / 255.0 - MEAN.reshape(3, 1, 1)) / STD.reshape(3, 1, 1))[None]\n    with torch.no_grad():\n        z = model(torch.from_numpy(np.ascontiguousarray(blob))).numpy()[0]\n    e = np.exp(z - z.max(axis=0, keepdims=True))\n    probs = e / e.sum(axis=0, keepdims=True)\n    return blob, z, np.stack([cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR) for p in probs])\n\n\ndef run(ckpt: str, onnx32: str, onnx16: str | None = None, samples: Path = ROOT / "public" / "samples",\n        n: int = 20, crop: int = 512, seed: int = 0, tol: float = 1e-3, tol16: float = 5e-2) -> list[dict]:\n    _, _, groups = load_class_map()\n    net, _ = load_checkpoint(ckpt)\n    model = DrishyamHead(net, groups).eval()\n    cs = crops(samples, n, crop, seed)\n    seg32 = OnnxSegmenter(onnx32)\n    assert np.allclose(seg32.mean.ravel(), MEAN) and np.allclose(seg32.std.ravel(), STD), "normalisation differs"\n    cases = [("fp32 accurate", seg32, "accurate", 512, tol), ("fp32 fast", seg32, "fast", 256, tol)]\n    if onnx16:\n        cases += [("fp16 accurate", OnnxSegmenter(onnx16), "accurate", 512, tol16),\n                  ("fp16 fast", OnnxSegmenter(onnx32, fp16_path=onnx16), "fast", 256, tol16)]\n    rows = []\n    for name, seg, mode, size, limit in cases:\n        dp = dz = zmax = 0.0\n        agree = []\n        dnn = seg.nets["fp16" if mode == "fast" else "accurate"]  # the net OnnxSegmenter.predict will use\n        for c in cs:\n            blob, z, p_ref = torch_reference(model, c, size)\n            s = seg.predict(c, mode)\n            dp = max(dp, float(np.abs(s.probs - p_ref).max()))\n            dnn.setInput(blob)\n            dz = max(dz, float(np.abs(np.asarray(dnn.forward())[0] - z).max()))\n            zmax = max(zmax, float(np.abs(z).max()))\n            agree.append(float((s.probs.argmax(0) == p_ref.argmax(0)).mean()))\n        rows.append({"case": name, "size": size, "max_dprob": dp, "max_dlogit": dz, "max_abs_logit": zmax,\n                     "label_agreement": float(np.mean(agree)), "tol": limit, "ok": dp <= limit})\n    return rows\n\n\ndef main(argv=None) -> int:\n    p = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    p.add_argument("--ckpt", required=True)\n    p.add_argument("--onnx", required=True, help="FP32 ONNX from export_onnx.py")\n    p.add_argument("--onnx-fp16")\n    p.add_argument("--samples", default=str(ROOT / "public" / "samples"))\n    p.add_argument("--n", type=int, default=20)\n    p.add_argument("--tol", type=float, default=1e-3, help="max |dprob| allowed for FP32")\n    p.add_argument("--tol-fp16", type=float, default=5e-2)\n    a = p.parse_args(argv)\n    rows = run(a.ckpt, a.onnx, a.onnx_fp16, Path(a.samples), a.n, tol=a.tol, tol16=a.tol_fp16)\n    print(f"OpenCV {cv2.__version__} DNN ENGINE_NEW vs torch {torch.__version__}, {a.n} crops")\n    print(f"{\'case\':<15}{\'size\':>5}{\'max|dprob|\':>12}{\'max|dlogit|\':>13}{\'max|logit|\':>12}{\'label agree\':>13}{\'tol\':>8}")\n    for r in rows:\n        print(f"{r[\'case\']:<15}{r[\'size\']:>5}{r[\'max_dprob\']:>12.2e}{r[\'max_dlogit\']:>13.2e}{r[\'max_abs_logit\']:>12.2f}"\n              f"{r[\'label_agreement\']:>13.5f}{r[\'tol\']:>8.0e}  {\'ok\' if r[\'ok\'] else \'FAIL\'}")\n    if not all(r["ok"] for r in rows if r["case"].startswith("fp32")):\n        print("FP32 over tolerance: do not install these weights")\n        return 1\n    if not all(r["ok"] for r in rows):\n        print("FP32 ok; FP16 over tolerance: install the FP32 ONNX alone (Drishyam\'s fast looks then use it too)")\n        return FP16_OVER\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n',
    'drishyam/__init__.py': '"""Drishyam: virtual-PTZ bridge inspector."""\n\n__version__ = "1.0.2"\n',
    'drishyam/imaging.py': '"""Image intake, pyramid, saliency overview and the virtual-PTZ gaze.\n\nThe camera moves (pyrDown, warpPerspective, resize) are stock OpenCV imgproc ops;\nKleidiCV HAL coverage of these calls is not measured. Saliency and the quality gate\nalso use CLAHE, adaptiveThreshold, Laplacian and connected components.\n"""\n\nfrom __future__ import annotations\n\nimport base64\nimport math\nimport struct\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\nMAX_PIXELS = 25_000_000\nMAX_DECODED_BYTES = 100_000_000  # what libpng/libjpeg may allocate; 16-bit RGBA at 25 MP would be 200 MB\nMIN_SIDE = 64\nLEVELS = 5  # L0 (full resolution) .. L4\nOVERVIEW_LEVEL = 3\nSHARPNESS_MIN = 12.0  # Laplacian variance below this is blur\nGLARE_MAX = 0.20  # fraction of blown-out pixels\nDARK_MAX = 0.50\n# Saliency\'s absolute null: each map is divided by max(its 99.5th percentile, floor), so the texture of a sound\n# surface is not stretched to full scale. Calibrated on bridge_08 and a 30-image seeded synthetic slice (git log).\nEDGE_FLOOR = 250.0  # Sobel magnitude of the CLAHE\'d 8-bit overview\nDARK_FLOOR = 70.0  # 9x9 blackhat, grey levels\nSAL_FLOOR = 0.55  # the weighted, blurred sum\nUNSEEN_W = 0.45  # blown-out or black patches: worth a look, so the quality gate can mark the area not assessed\nUNSEEN_MAX_FRAC = 0.05  # a blown-out or black area larger than this share of the image gets no look (sky, a shadow):\nUNSEEN_RIM = 10  # it and this many overview px around it (its edge) score 0; the work order marks it not assessed\n\n_SOF = {0xC0, 0xC1, 0xC2, 0xC3, 0xC5, 0xC6, 0xC7, 0xC9, 0xCA, 0xCB, 0xCD, 0xCE, 0xCF}\n\n\nclass ImageError(ValueError):\n    """The upload is not an image we accept."""\n\n\n_PNG_CHANNELS = {0: 1, 2: 3, 3: 3, 4: 2, 6: 4}  # colour type -> channels after decode (palette expands)\n\n\ndef sniff(data: bytes) -> tuple[int, int, float]:\n    """Width, height and decoded bytes per pixel from the header, before any pixel is decoded."""\n    if data[:8] == b"\\x89PNG\\r\\n\\x1a\\n":\n        if len(data) < 26 or data[12:16] != b"IHDR":\n            raise ImageError("corrupt PNG header")\n        w, h = struct.unpack(">II", data[16:24])\n        depth, colour = data[24], data[25]\n        if colour not in _PNG_CHANNELS or depth not in (1, 2, 4, 8, 16):\n            raise ImageError("unsupported PNG colour type or bit depth")\n        return w, h, _PNG_CHANNELS[colour] * max(depth, 8) / 8\n    if data[:3] == b"\\xff\\xd8\\xff":\n        i = 2\n        while i + 9 <= len(data):\n            if data[i] != 0xFF:\n                raise ImageError("corrupt JPEG marker stream")\n            marker = data[i + 1]\n            if marker == 0xFF:  # fill byte\n                i += 1\n                continue\n            if marker in (0xD8, 0x01) or 0xD0 <= marker <= 0xD7:\n                i += 2\n                continue\n            (seg_len,) = struct.unpack(">H", data[i + 2 : i + 4])\n            if seg_len < 2:\n                raise ImageError("corrupt JPEG segment")\n            if marker in _SOF:\n                precision = data[i + 4]\n                h, w = struct.unpack(">HH", data[i + 5 : i + 9])\n                comps = data[i + 9] if i + 9 < len(data) else 3\n                return w, h, max(comps, 3) * max(precision, 8) / 8\n            i += 2 + seg_len\n        raise ImageError("JPEG has no frame header")\n    raise ImageError("only JPEG and PNG images are accepted")\n\n\ndef sniff_size(data: bytes) -> tuple[int, int]:\n    return sniff(data)[:2]\n\n\ndef decode_image(data: bytes, max_pixels: int = MAX_PIXELS) -> np.ndarray:\n    """Decode an upload to BGR, refusing decompression bombs before decoding."""\n    if not data:\n        raise ImageError("empty upload")\n    w, h, bpp = sniff(data)\n    if w < MIN_SIDE or h < MIN_SIDE:\n        raise ImageError(f"image is {w}x{h}; the minimum side is {MIN_SIDE} px")\n    if w * h > max_pixels:\n        raise ImageError(f"image is {w}x{h} = {w * h / 1e6:.1f} MP; the limit is {max_pixels / 1e6:.0f} MP")\n    if w * h * bpp > MAX_DECODED_BYTES:\n        raise ImageError(f"image would decode to {w * h * bpp / 1e6:.0f} MB; the limit is "\n                         f"{MAX_DECODED_BYTES / 1e6:.0f} MB (use 8-bit colour)")\n    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)\n    # EXIF orientation may swap the axes; anything else means the header lied.\n    if img is None or sorted(img.shape[:2]) != sorted((h, w)):\n        raise ImageError("image data could not be decoded")\n    return img\n\n\ndef build_pyramid(img: np.ndarray, levels: int = LEVELS) -> list[np.ndarray]:\n    pyr = [img]\n    for _ in range(levels - 1):\n        pyr.append(cv2.pyrDown(pyr[-1]))\n    return pyr\n\n\ndef _norm(x: np.ndarray, floor: float = 0.0) -> np.ndarray:\n    """x / max(p99.5, floor), clipped to 0-1. The floor is an absolute null: without it a sound surface\'s own\n    texture is stretched to full scale and always yields candidates."""\n    hi = max(float(np.percentile(x, 99.5)), floor)\n    return np.clip(x / hi, 0, 1) if hi > 1e-6 else np.zeros_like(x, np.float32)\n\n\ndef _clahe():\n    """A new CLAHE per call: the object keeps its working buffers, so one shared by concurrent\n    requests (web.py runs them in a threadpool) mixes their images. Creating one costs microseconds."""\n    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))\n\n\ndef saliency(img: np.ndarray) -> np.ndarray:\n    """0-1 prior of \'something worth a closer look\', built from imgproc ops only."""\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    eq = cv2.GaussianBlur(_clahe().apply(gray), (3, 3), 0)\n    gx = cv2.Sobel(eq, cv2.CV_32F, 1, 0, ksize=3)\n    gy = cv2.Sobel(eq, cv2.CV_32F, 0, 1, ksize=3)\n    edges = _norm(cv2.magnitude(gx, gy), EDGE_FLOOR)\n    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))\n    thin_dark = _norm(cv2.morphologyEx(eq, cv2.MORPH_BLACKHAT, kernel).astype(np.float32), DARK_FLOOR)\n    local_dark = cv2.adaptiveThreshold(eq, 1, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 21, 12)\n    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)\n    rust = cv2.inRange(hsv, (3, 70, 35), (25, 255, 235)).astype(np.float32) / 255\n    white = cv2.inRange(hsv, (0, 0, 205), (180, 40, 255)).astype(np.float32) / 255\n    small, large, _ = _unseen(gray)\n    s = 0.30 * edges + 0.40 * thin_dark + 0.10 * local_dark + 0.45 * rust + 0.15 * white + UNSEEN_W * small\n    s = cv2.GaussianBlur(s, (0, 0), 2.0)\n    if large.any():  # its edge is no defect either: zeroed out to the 21-px adaptive block\'s reach\n        s[cv2.dilate(large.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * UNSEEN_RIM + 1,) * 2)) > 0] = 0\n    return _norm(s, SAL_FLOOR).astype(np.float32)\n\n\ndef _unseen(gray: np.ndarray) -> tuple[np.ndarray, np.ndarray, list]:\n    """Blown-out or black pixels (the quality gate\'s glare and dark), split by connected area: the patches small\n    enough to be worth a look (0/1 float), the areas over UNSEEN_MAX_FRAC of the image (bool), and those areas as\n    (x, y, w, h, reason)."""\n    m = 255 - cv2.inRange(gray, 6, 249)\n    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)\n    big = [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] > UNSEEN_MAX_FRAC * gray.size]\n    large = np.isin(lab, big)\n    areas = [(*(int(v) for v in st[i, :4]), "glare" if gray[lab == i].mean() > 127 else "dark") for i in big]\n    return ((m > 0) & ~large).astype(np.float32), large, areas\n\n\ndef unseen_areas(ov: np.ndarray, scale: int, l0_size: tuple[int, int]) -> list[dict]:\n    """Work-order `not_assessed` entries for the blown-out or black areas of the overview `ov` that are too large to\n    spend looks on (`saliency` scores them 0): step 0 (the overview), the L0 box, reason glare or dark."""\n    to_l0 = np.diag([float(scale), float(scale), 1.0])\n    return [{"step": 0, "box": l0_box((x, y, w, h), to_l0, l0_size), "reason": reason, "action": "re-shoot"}\n            for x, y, w, h, reason in _unseen(cv2.cvtColor(ov, cv2.COLOR_BGR2GRAY))[2]]\n\n\n@dataclass\nclass Candidate:\n    id: str\n    cx: int  # L0 centre\n    cy: int\n    box: tuple[int, int, int, int]  # L0 window of the first look\n    prior: float\n\n\ndef box_iou(a, b) -> float:\n    ax, ay, aw, ah = a\n    bx, by, bw, bh = b\n    iw = max(0, min(ax + aw, bx + bw) - max(ax, bx))\n    ih = max(0, min(ay + ah, by + bh) - max(ay, by))\n    inter = iw * ih\n    return inter / float(aw * ah + bw * bh - inter) if inter else 0.0\n\n\ndef find_candidates(\n    sal: np.ndarray, level: int, window_l0: int, l0_size: tuple[int, int], max_n: int = 20, thresh: float = 0.5\n) -> list[Candidate]:\n    """Ranked candidate windows: salient blobs at the overview level, split into first-look windows."""\n    scale = 2**level\n    win = max(4, window_l0 // scale)\n    mask = cv2.dilate((sal >= thresh).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)\n    W, H = l0_size\n    raw = []\n    for i in range(1, n):\n        x, y, w, h, area = stats[i]\n        if area < 4:\n            continue\n        step = max(1, int(win * 0.75))\n        xs = range(x + min(w, win) // 2, x + w + 1, step) if w > win else [x + w // 2]\n        ys = range(y + min(h, win) // 2, y + h + 1, step) if h > win else [y + h // 2]\n        for cy in ys:\n            for cx in xs:\n                x0, y0 = max(0, cx - win // 2), max(0, cy - win // 2)\n                tile = sal[y0 : cy + win // 2, x0 : cx + win // 2]\n                tile_lab = labels[y0 : cy + win // 2, x0 : cx + win // 2]\n                inside = tile[tile_lab == i]\n                if inside.size == 0:\n                    continue\n                # Re-centre on the strongest pixel of this blob inside the tile.\n                ty, tx = np.unravel_index(np.argmax(np.where(tile_lab == i, tile, -1)), tile.shape)\n                px = int(min(max((x0 + tx) * scale, 0), W - 1))\n                py = int(min(max((y0 + ty) * scale, 0), H - 1))\n                raw.append((float(inside.mean()), py, px))  # mean: max saturates at 1 after _norm\n    raw.sort(key=lambda t: (-t[0], t[1], t[2]))\n    kept: list[Candidate] = []\n    for prior, py, px in raw:\n        box = (px - window_l0 // 2, py - window_l0 // 2, window_l0, window_l0)\n        if any(box_iou(box, k.box) > 0.3 for k in kept):\n            continue\n        kept.append(Candidate(f"r{len(kept) + 1}", px, py, box, round(prior, 4)))\n        if len(kept) >= max_n:\n            break\n    return kept\n\n\ndef _rotation(yaw_deg: float, pitch_deg: float) -> np.ndarray:\n    y, p = math.radians(yaw_deg), math.radians(pitch_deg)\n    ry = np.array([[math.cos(y), 0, math.sin(y)], [0, 1, 0], [-math.sin(y), 0, math.cos(y)]])\n    rx = np.array([[1, 0, 0], [0, math.cos(p), -math.sin(p)], [0, math.sin(p), math.cos(p)]])\n    return ry @ rx\n\n\ndef gaze(\n    pyr: list[np.ndarray], x: float, y: float, level: int, yaw: float = 0.0, pitch: float = 0.0, size: int = 512\n) -> tuple[np.ndarray, np.ndarray]:\n    """Render a virtual-PTZ view centred on L0 point (x, y) at a pyramid level.\n\n    Returns the crop and the 3x3 matrix mapping crop pixels to L0 pixels. With\n    yaw = pitch = 0 the view is a pure translation, so the mapping is exact.\n    """\n    if not 0 <= level < len(pyr):\n        raise ValueError(f"level must be 0..{len(pyr) - 1}")\n    s = 2.0**level\n    f = float(size)\n    k = np.array([[f, 0, size / 2], [0, f, size / 2], [0, 0, 1]])\n    t = np.array([[1, 0, x / s - size / 2], [0, 1, y / s - size / 2], [0, 0, 1]])\n    m = t @ k @ _rotation(yaw, pitch) @ np.linalg.inv(k)\n    crop = cv2.warpPerspective(\n        pyr[level], m, (size, size), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REFLECT_101\n    )\n    to_l0 = np.diag([s, s, 1.0]) @ m\n    return crop, to_l0\n\n\ndef map_points(to_l0: np.ndarray, pts) -> np.ndarray:\n    p = np.asarray(pts, np.float64).reshape(-1, 1, 2)\n    return cv2.perspectiveTransform(p, to_l0).reshape(-1, 2)\n\n\ndef footprint(to_l0: np.ndarray, l0_size: tuple[int, int], size: int) -> np.ndarray:\n    """(size, size) bool: crop pixels that show the image, i.e. map inside the W x H L0 frame. Where a view\n    runs past the image, `gaze` fills it by reflection, so anything seen there is a mirror image."""\n    W, H = l0_size\n    h = to_l0\n    if np.allclose(h[[0, 1, 2, 2], [1, 0, 0, 1]], 0):  # yaw = pitch = 0: a scaled translation, rows x columns\n        u = np.arange(size)\n        x, y = (h[0, 0] * u + h[0, 2]) / h[2, 2], (h[1, 1] * u + h[1, 2]) / h[2, 2]\n        return ((y >= 0) & (y <= H - 1))[:, None] & ((x >= 0) & (x <= W - 1))[None, :]\n    v, u = np.mgrid[:size, :size]\n    x, y = map_points(h, np.stack([u.ravel(), v.ravel()], 1)).T\n    return ((x >= 0) & (x <= W - 1) & (y >= 0) & (y <= H - 1)).reshape(size, size)\n\n\ndef l0_box(rect, to_l0: np.ndarray, l0_size: tuple[int, int]) -> list[int]:\n    """Crop rect (x, y, w, h) as an L0 box [x, y, w, h], clipped to the W x H image."""\n    x, y, w, h = rect\n    # All four corners: under yaw/pitch the view is a perspective warp, so two corners can swap order.\n    c = map_points(to_l0, [[x, y], [x + w, y], [x, y + h], [x + w, y + h]])\n    hi = np.asarray(l0_size, float)\n    (x0, y0), (x1, y1) = np.clip(c.min(0), 0.0, hi), np.clip(c.max(0), 0.0, hi)\n    return [int(x0), int(y0), int(x1 - x0), int(y1 - y0)]\n\n\ndef quality(crop: np.ndarray) -> dict:\n    """Sharpness (Laplacian variance) and exposure gate; a failed gate means \'do not trust this view\'."""\n    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)\n    sharp = float(cv2.Laplacian(gray, cv2.CV_32F).var())\n    glare = float(np.count_nonzero(gray >= 250)) / gray.size\n    dark = float(np.count_nonzero(gray <= 5)) / gray.size\n    reason = None\n    if glare > GLARE_MAX:\n        reason = "glare"\n    elif dark > DARK_MAX:\n        reason = "dark"\n    elif sharp < SHARPNESS_MIN:\n        reason = "blur"\n    return {"ok": reason is None, "reason": reason, "sharpness": round(sharp, 1),\n            "exposure": round(float(gray.mean()) / 255, 3)}\n\n\ndef enhance(crop: np.ndarray) -> np.ndarray:\n    """CLAHE on the lightness channel: a cheaper second look before zooming further."""\n    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)\n    lab[..., 0] = _clahe().apply(lab[..., 0])\n    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)\n\n\ndef jpeg_b64(img: np.ndarray, max_side: int, quality_: int = 80) -> str:\n    h, w = img.shape[:2]\n    k = max_side / max(h, w)\n    if k < 1:\n        img = cv2.resize(img, (max(1, round(w * k)), max(1, round(h * k))), interpolation=cv2.INTER_AREA)\n    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality_])\n    return base64.b64encode(buf.tobytes()).decode() if ok else ""\n\n\ndef png_b64(gray: np.ndarray) -> str:\n    ok, buf = cv2.imencode(".png", gray)\n    return base64.b64encode(buf.tobytes()).decode() if ok else ""\n',
    'drishyam/measure.py': '"""Crack geometry in pixels: length, mean width, area and orientation.\n\nThe centreline comes from hit-or-miss thinning (stock OpenCV imgproc ops; KleidiCV\nHAL coverage of these calls is not measured); component labelling and contours are\nplain imgproc. Length is the smoothed arc length of that 1-px line and\nmean width is area / length. Everything is reported in L0 image pixels; there\nare no millimetres anywhere in Drishyam.\n"""\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\n_BASE = ([[-1, -1, -1], [0, 1, 0], [1, 1, 1]], [[0, -1, -1], [1, 1, -1], [0, 1, 0]])\n_THIN_KERNELS = [np.rot90(np.array(k, np.int8), r).copy() for k in _BASE for r in range(4)]\n\n\ndef thin(mask: np.ndarray) -> np.ndarray:\n    """1-px 8-connected centreline of a 0/1 mask (classic hit-or-miss thinning)."""\n    img = (mask > 0).astype(np.uint8)\n    while True:\n        before = cv2.countNonZero(img)\n        for k in _THIN_KERNELS:\n            img &= ~cv2.morphologyEx(img, cv2.MORPH_HITMISS, k)\n        if cv2.countNonZero(img) == before:\n            return img\n\n\ndef measure(mask: np.ndarray, scale: float = 1.0, linear: bool = True) -> dict:\n    """Geometry of the largest component of `mask`, scaled from crop to L0 pixels.\n\n    `linear` (cracks) measures length along the thinned centreline; for area defects\n    (spalling, stains) length is the long side of the minimum-area rectangle, which\n    skips thinning a large blob, by far the most expensive step.\n    """\n    m = (mask > 0).astype(np.uint8)\n    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)\n    if n < 2:\n        return {"length_px": 0.0, "width_px": 0.0, "area_px": 0, "orientation_deg": 0.0}\n    biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    x, y, w, h = stats[biggest, :4]\n    comp = (lab[y : y + h, x : x + w] == biggest).astype(np.uint8)  # thin inside the bounding box only\n    area = int(stats[biggest, cv2.CC_STAT_AREA])\n    (_, _), (rw, rh), angle = cv2.minAreaRect(cv2.findNonZero(comp))\n    if linear:\n        padded = cv2.copyMakeBorder(comp, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)\n        contours, _ = cv2.findContours(thin(padded), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n        # A closed contour around a 1-px curve walks it twice; approxPolyDP removes the\n        # staircase bias of 8-connected steps (about +8% on a 30-degree line otherwise).\n        length = max(1.0, sum(cv2.arcLength(cv2.approxPolyDP(c, 1.0, True), True) for c in contours) / 2)\n    else:\n        length = max(1.0, float(max(rw, rh)))\n    if rw < rh:\n        angle += 90.0\n    return {\n        "length_px": round(length * scale, 1),\n        "width_px": round(area / length * scale, 2),\n        "area_px": round(area * scale * scale),\n        "orientation_deg": round(angle % 180.0, 1),\n    }\n',
    'drishyam/segment.py': '"""Per-class defect segmentation of a gaze crop, with uncertainty.\n\nTwo engines share one output: `ClassicalSegmenter` (imgproc evidence maps, no\nweights, runs anywhere) and `OnnxSegmenter` (a dacl10k-trained model on the\nOpenCV 5 DNN `ENGINE_NEW`). The policy only reads probabilities, areas and\nentropy, so swapping engines changes no other code.\n"""\n\nfrom __future__ import annotations\n\nimport functools\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\n\nfrom . import imaging\nfrom .measure import thin\n\nCLASSES = ("background", "crack", "spalling", "efflorescence", "exposed_rebar", "corrosion")\nDEFECTS = CLASSES[1:]\nMIN_BLOB_PX = 12  # connected components smaller than this are noise\nENTROPY_MIN_PX = 64  # fewer ambiguous pixels than this is not uncertainty worth a closer look\nDEFECT_PX_BG_MAX = 0.7  # a pixel counts toward entropy when p(background) is below this\nCRACK_LINK_SIGMA = 2.5  # crack hysteresis low threshold: half the 5-sigma seed threshold (Canny\'s 2:1)\nSTRAIGHT_RMS_PX = 1.5  # crack centreline RMS residual from its cv2.fitLine (Huber) line, L0 px\nSTRAIGHT_ASPECT = 20.0  # length over mean width, and the minAreaRect long side over its short side\nSTRAIGHT_CAUSE = "straight line: formwork or joint"\nPORE_ASPECT = 8.0  # a crack is drafted only if its measured length is at least this many mean widths: not a blob\nPORE_CAUSE = "not elongated: pore or bug hole"\n\n\n@dataclass\nclass Segmentation:\n    probs: np.ndarray  # (C, H, W) float32, sums to 1 over C\n    labels: np.ndarray  # (H, W) uint8 class index\n    entropy: float  # mean binary damage-vs-background entropy over defect-like pixels, 0..1\n    areas: dict[str, int]  # crop pixels per defect class\n    confidence: dict[str, float]  # mean probability over each class\'s pixels\n    latency_ms: float\n    model: str\n    net_px: int = 0  # pixels the engine computed on: the net\'s input tensor (pad included), or the classical work image\n    lines: list = field(default_factory=list)  # crack components dropped as straight lines (`drop_straight_lines`)\n\n    def mask(self, cls: str) -> np.ndarray:\n        return (self.labels == CLASSES.index(cls)).astype(np.uint8)\n\n    @property\n    def best(self) -> str | None:\n        present = {c: a * self.confidence[c] for c, a in self.areas.items() if a}\n        return max(present, key=present.get) if present else None\n\n\ndef _robust(x: np.ndarray) -> tuple[float, float]:\n    """Median and MAD-sigma of a map, from a strided sample (the map is a texture, not a signal)."""\n    sample = x[::4, ::4]\n    med = float(np.median(sample))\n    return med, max(1.0, 1.4826 * float(np.median(np.abs(sample - med))))\n\n\ndef _wide_blur(x: np.ndarray, sigma: float) -> np.ndarray:\n    """Large-sigma Gaussian on a 1/8 copy: the same low-pass at a fraction of the cost."""\n    h, w = x.shape[:2]\n    small = cv2.resize(x, (max(1, w // 8), max(1, h // 8)), interpolation=cv2.INTER_AREA)\n    return cv2.resize(cv2.GaussianBlur(small, (0, 0), sigma / 8), (w, h), interpolation=cv2.INTER_LINEAR)\n\n\ndef _hysteresis(strong: np.ndarray, weak: np.ndarray, min_len: int) -> np.ndarray:\n    """Canny-style linking for a linear defect: a pixel whose `weak` evidence passes the coin\n    flip counts as crack when it is 8-connected through such pixels to one whose `strong`\n    evidence does, and the linked trace spans at least `min_len` px (a line, not a speck).\n    A faint stretch of a crack then stays part of the crack at every scale, while faint\n    texture around an isolated dark speck stays background."""\n    n, lab, stats, _ = cv2.connectedComponentsWithStats((weak > 0.5).astype(np.uint8), connectivity=8)\n    keep = np.zeros(n, bool)\n    keep[lab[strong > 0.5]] = True\n    keep &= np.maximum(stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT]) >= min_len\n    keep[0] = False\n    return np.where(keep[lab], np.maximum(strong, weak), strong)\n\n\ndef _softmax(logits: np.ndarray) -> np.ndarray:\n    e = np.exp(logits - logits.max(axis=0, keepdims=True))\n    return e / e.sum(axis=0, keepdims=True)\n\n\ndef _clean(labels: np.ndarray, probs: np.ndarray, ids) -> tuple[dict[str, int], dict[str, float]]:\n    """Drop specks of the classes `ids` from `labels` (in place); their areas and mean confidences."""\n    areas, conf = {}, {}\n    for ci in ids:\n        m = (labels == ci).astype(np.uint8)\n        _, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)\n        small = stats[:, cv2.CC_STAT_AREA] < MIN_BLOB_PX\n        small[0] = False\n        if small.any():\n            drop = small[lab]\n            labels[drop] = 0\n            m[drop] = 0\n        a = int(np.count_nonzero(m))\n        areas[CLASSES[ci]] = a\n        conf[CLASSES[ci]] = round(float(probs[ci][m > 0].mean()), 4) if a else 0.0\n    return areas, conf\n\n\ndef _entropy(probs: np.ndarray) -> float:\n    """Uncertainty = binary entropy of "is this damage", averaged where damage is plausible.\n    1.0 is a coin flip; a crisp mask scores ~0.2 (its anti-aliased rim)."""\n    p_def = 1.0 - probs[0]\n    defect_like = p_def > 1.0 - DEFECT_PX_BG_MAX\n    if np.count_nonzero(defect_like) < ENTROPY_MIN_PX:\n        return 0.0\n    q = np.clip(p_def[defect_like], 1e-6, 1 - 1e-6)\n    return round(float((-(q * np.log2(q) + (1 - q) * np.log2(1 - q))).mean()), 4)\n\n\ndef finish(probs: np.ndarray, t0: float, model: str, net_px: int | None = None) -> Segmentation:\n    """Labels, areas, confidences and entropy from a (C, H, W) probability stack. `net_px`: the pixels the engine\n    computed on (default: the stack\'s own H x W)."""\n    labels = probs.argmax(axis=0).astype(np.uint8)\n    areas, conf = _clean(labels, probs, range(1, len(CLASSES)))\n    return Segmentation(probs, labels, _entropy(probs), areas, conf, (time.perf_counter() - t0) * 1000, model,\n                        probs[0].size if net_px is None else net_px)\n\n\ndef restrict(seg: Segmentation, valid: np.ndarray) -> Segmentation:\n    """`seg` with background forced wherever `valid` is False (crop pixels that are not the image, see\n    `imaging.footprint`), so labels, areas, confidence, entropy and everything built on them see no\n    mirrored defect. Equal to `finish` on the masked probabilities, at a third of its cost: inside `valid`\n    the argmax is unchanged, so only classes labelled outside it need their specks re-checked."""\n    if valid.all():\n        return seg\n    t0 = time.perf_counter()\n    probs = seg.probs * valid\n    probs[0] += ~valid\n    labels = np.where(valid, seg.labels, 0).astype(np.uint8)\n    areas, conf = dict(seg.areas), dict(seg.confidence)\n    a, c = _clean(labels, probs, [int(ci) for ci in np.unique(seg.labels[~valid]) if ci])\n    areas.update(a)\n    conf.update(c)\n    return Segmentation(probs, labels, _entropy(probs), areas, conf,\n                        seg.latency_ms + (time.perf_counter() - t0) * 1000, seg.model, seg.net_px)\n\n\ndef drop_straight_lines(seg: Segmentation, scale: float = 1.0) -> Segmentation:\n    """`seg` without the crack components that are one straight line, each listed in `.lines` as ((x, y, w, h) in\n    the view, then length, mean width and residual in L0 px; `scale` is L0 px per view px). A centreline within\n    STRAIGHT_RMS_PX L0 px (RMS) of its cv2.fitLine line over a length of at least STRAIGHT_ASPECT x its mean width,\n    inside a minAreaRect at least that thin, is a formwork seam, joint or edge: real cracks meander. The residual\n    is judged in L0 px because a coarse view smooths a crack\'s jitter away, so only a fine view can call a line\n    straight. Its pixels become background (`restrict`): no finding, entropy or escalation comes from it."""\n    n, lab, stats, _ = cv2.connectedComponentsWithStats(seg.mask("crack"), connectivity=8)\n    lines, keep = [], np.ones(seg.labels.shape, bool)\n    for i in range(1, n):\n        x, y, w, h, area = (int(v) for v in stats[i])\n        comp = (lab[y : y + h, x : x + w] == i).astype(np.uint8)\n        (_, _), (rw, rh), _ = cv2.minAreaRect(cv2.findNonZero(comp))\n        length = max(rw, rh)\n        if length < STRAIGHT_ASPECT * max(1.0, min(rw, rh)) or length * length < STRAIGHT_ASPECT * area:\n            continue  # not long and thin, the common case: decided before any thinning\n        c = cv2.findNonZero(thin(cv2.copyMakeBorder(comp, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)))\n        c = c.reshape(-1, 2).astype(np.float32)\n        vx, vy, x0, y0 = cv2.fitLine(c, cv2.DIST_HUBER, 0, 0.01, 0.01).ravel()\n        # RMS over the 90 % of centreline points nearest the line: thinning leaves spurs where a speck touches it\n        d = np.sort(np.abs((c[:, 0] - x0) * vy - (c[:, 1] - y0) * vx))[: max(1, len(c) * 9 // 10)]\n        rms = scale * float(np.sqrt(np.mean(d * d)))\n        if rms < STRAIGHT_RMS_PX:\n            keep[lab == i] = False\n            lines.append(((x, y, w, h), round(length * scale, 1), round(area / length * scale, 2), round(rms, 2)))\n    if not lines:\n        return seg\n    out = restrict(seg, keep)\n    out.lines = lines\n    return out\n\n\ndef rejected_lines(seg: Segmentation, to_l0: np.ndarray, l0_size: tuple[int, int]) -> list[dict]:\n    """`seg.lines` as trace records (the step\'s `rejected` list): class, cause, L0 box, length, width, residual."""\n    return [{"cls": "crack", "cause": STRAIGHT_CAUSE, "box": imaging.l0_box(b, to_l0, l0_size), "length_px": ln,\n             "width_px": wd, "residual_px": rms} for b, ln, wd, rms in seg.lines]\n\n\ndef elongated(m: dict) -> bool:\n    """A crack `measure` that is a line, not a blob: length >= PORE_ASPECT x mean width (a pore or bug hole is not)."""\n    return m["length_px"] >= PORE_ASPECT * m["width_px"]\n\n\ndef segment_view(segmenter, crop: np.ndarray, model: str, valid: np.ndarray, scale: float = 1.0) -> Segmentation:\n    """Segment a gaze crop, keeping only what lies inside the image (`valid`, from `imaging.footprint`) and is not\n    a straight line (`drop_straight_lines`; `scale`: L0 px per crop px, 2**level)."""\n    return drop_straight_lines(restrict(segmenter.predict(crop, model), valid), scale)\n\n\nclass ClassicalSegmenter:\n    """Hand-built evidence maps turned into calibrated class probabilities.\n\n    No learned weights: this is the honest baseline that keeps the agent runnable\n    with zero downloads. Gains map evidence 0.5 to a coin flip against background,\n    which is where the entropy the policy escalates on comes from.\n    """\n\n    name = "classical-v1"\n    gain = 9.0\n\n    def evidence(self, crop: np.ndarray) -> np.ndarray:\n        """Five 0-1 maps. Thresholds are relative to the crop\'s own texture, so a rough\n        but healthy surface does not read as damage."""\n        h, w = crop.shape[:2]\n        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)\n        # A CLAHE per call: it keeps working buffers, and default_segmenter() is shared by request threads.\n        g = cv2.GaussianBlur(cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray), (3, 3), 0)\n        k = max(7, (min(h, w) // 34) | 1)\n        bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))\n        bh = bh.astype(np.float32)\n        med, noise = _robust(bh)\n        crack = np.clip((bh - med - 3.0 * noise) / (4.0 * noise), 0, 1)  # coin flip at 5 sigma\n        # The same ramp with its coin flip at CRACK_LINK_SIGMA: evidence a crack pixel may have\n        # when it continues a confident one (hysteresis, below).\n        linked = np.clip((bh - med - (CRACK_LINK_SIGMA - 2.0) * noise) / (4.0 * noise), 0, 1)\n\n        gf = gray.astype(np.float32)\n        sig = max(3.0, min(h, w) / 60)\n        mu = cv2.GaussianBlur(gf, (0, 0), sig)\n        rough = np.sqrt(np.maximum(cv2.GaussianBlur(gf * gf, (0, 0), sig) - mu * mu, 0))\n        r_med, r_noise = _robust(rough)\n        # Spalling is a wide, dark, rough patch. Darkness is read from a wide median (which\n        # erases cracks up to ~k px wide) and the map is opened with a kernel wider than any\n        # crack, so neither a crack\'s contrast nor the rim of a stain can pass as a spall.\n        wide = 2 * k + 1\n        cf = cv2.GaussianBlur(cv2.medianBlur(gray, wide).astype(np.float32), (0, 0), sig)\n        darker = np.clip((_wide_blur(gf, sig * 6) - cf) / 25.0, 0, 1)\n        spall_raw = np.clip((rough - r_med - 2.5 * r_noise) / (3.0 * r_noise), 0, 1) * np.clip(2.5 * darker, 0, 1)\n        open_k = cv2.getStructuringElement(cv2.MORPH_RECT, (wide, wide))  # rect: separable, fast\n\n        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)\n        rust = cv2.GaussianBlur(\n            cv2.inRange(hsv, (3, 70, 35), (25, 255, 235)).astype(np.float32) / 255, (0, 0), 2.0\n        )\n        spall = cv2.morphologyEx(spall_raw * (1 - rust), cv2.MORPH_OPEN, open_k)\n        near_spall = cv2.dilate(spall, cv2.getStructuringElement(cv2.MORPH_RECT, (k * 3, k * 3)))\n        rebar = rust * np.clip(2 * near_spall, 0, 1)\n        corrosion = rust * (1 - np.clip(2 * near_spall, 0, 1))\n\n        v = hsv[..., 2].astype(np.float32)\n        lift = v - _wide_blur(v, sig * 4)\n        white = (hsv[..., 1] < 40).astype(np.float32)\n        efflo = np.clip((lift - 22.0) / 25.0, 0, 1) * white\n        efflo = cv2.morphologyEx(efflo, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))\n\n        # A thin dark line inside spalling is the spall\'s own edge, and rust inside it is the bar.\n        # Linking never runs through rust (a faint dark line there is the bar\'s or the stain\'s edge),\n        # and a linked trace must be longer than k, the widest dark line the blackhat passes.\n        not_spall = 1 - np.clip(1.5 * spall, 0, 1)\n        crack = _hysteresis(crack * not_spall, linked * not_spall * (1 - rust), k)\n        spall = spall * (1 - np.clip(1.5 * rebar, 0, 1))\n        return np.stack([crack, spall, efflo, rebar, corrosion]).astype(np.float32)\n\n    def predict(self, crop: np.ndarray, model: str = "accurate") -> Segmentation:\n        t0 = time.perf_counter()\n        h, w = crop.shape[:2]\n        work = crop if model == "accurate" else cv2.resize(crop, (w // 2, h // 2), interpolation=cv2.INTER_AREA)\n        ev = self.evidence(work)\n        logits = np.concatenate([np.zeros((1, *ev.shape[1:]), np.float32), self.gain * (ev - 0.5)])\n        probs = _softmax(logits)\n        if work is not crop:\n            probs = np.stack([cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR) for p in probs])\n        return finish(probs, t0, model, work.shape[0] * work.shape[1])\n\n\nclass OnnxSegmenter:\n    """A segmentation ONNX (N, C, H, W logits) on the OpenCV 5 DNN new engine.\n\n    Each crop is letterboxed into the square `input_size` input (`input_size // 2` for the fast model): one scale\n    for both axes, the rest padded with the normalised mean (zero). The logits are cropped back to the letterbox\'s\n    content before they are resized to the crop, so a non-square crop is never squashed.\n    `class_map[i]` is the Drishyam class index that model channel i adds to, so a\n    19-class dacl10k head collapses onto CODEBRIM\'s five defects plus background.\n    """\n\n    def __init__(self, path: str, fp16_path: str | None = None, input_size: int = 512,\n                 class_map: list[int] | None = None, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):\n        self.name = f"onnx:{path.rsplit(\'/\', 1)[-1]}"\n        self.files = (path, fp16_path)  # what the nets were read from (evals.harness hashes them)\n        self.nets = {"accurate": self._load(path)}\n        self.nets["fp16"] = self._load(fp16_path) if fp16_path else self.nets["accurate"]\n        self.size = input_size\n        self.class_map = class_map\n        self.mean = np.array(mean, np.float32).reshape(3, 1, 1)\n        self.std = np.array(std, np.float32).reshape(3, 1, 1)\n        # A Net holds its input between setInput and forward, and default_segmenter() is shared by\n        # request threads. ponytail: one lock for both nets; a lock per net if fp16 and accurate must overlap.\n        self.lock = threading.Lock()\n\n    @staticmethod\n    def _load(path: str):\n        return cv2.dnn.readNetFromONNX(path, engine=cv2.dnn.ENGINE_NEW)\n\n    def predict(self, crop: np.ndarray, model: str = "accurate") -> Segmentation:\n        t0 = time.perf_counter()\n        h, w = crop.shape[:2]\n        size = self.size if model == "accurate" else self.size // 2\n        k = size / max(h, w)\n        nw, nh = max(1, round(w * k)), max(1, round(h * k))\n        rgb = cv2.cvtColor(cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)\n        blob = np.zeros((1, 3, size, size), np.float32)\n        blob[0, :, :nh, :nw] = (rgb.astype(np.float32).transpose(2, 0, 1) / 255.0 - self.mean) / self.std\n        net = self.nets["fp16" if model in ("fp16", "fast") else "accurate"]\n        with self.lock:\n            net.setInput(blob)\n            logits = np.asarray(net.forward())[0].astype(np.float32)\n        oh, ow = logits.shape[1:]  # the output may be strided: crop the content at the same fraction\n        probs = _softmax(logits[:, : max(1, round(nh * oh / size)), : max(1, round(nw * ow / size))])\n        if self.class_map is not None:\n            merged = np.zeros((len(CLASSES), *probs.shape[1:]), np.float32)\n            for ch, target in enumerate(self.class_map):\n                merged[target] += probs[ch]\n            probs = merged\n        if probs.shape[0] != len(CLASSES):\n            raise ValueError(f"model has {probs.shape[0]} classes; pass class_map to reach {len(CLASSES)}")\n        probs = np.stack([cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR) for p in probs])\n        return finish(probs, t0, model, size * size)\n\n\ndef mask_iou(a: np.ndarray, b: np.ndarray) -> float:\n    inter = np.count_nonzero(a & b)\n    union = np.count_nonzero(a | b)\n    return inter / union if union else 1.0\n\n\n@functools.cache\ndef default_segmenter():\n    """The ONNX model named by DRISHYAM_ONNX (optional DRISHYAM_ONNX_FP16), else the classical engine."""\n    path = os.environ.get("DRISHYAM_ONNX")\n    return OnnxSegmenter(path, os.environ.get("DRISHYAM_ONNX_FP16")) if path else ClassicalSegmenter()\n',
}
SOURCE_SHA256 = "822ceb0f9aa51b42e4fad9cbc7c7ff914e3d5934ea49363a111aa19f237a1af0"  # make_notebook.py writes the sha256 of the sources it built the notebook from
NOTEBOOK = "__file__" not in globals()
KAGGLE = Path("/kaggle/working").is_dir()
CV5 = Path(tempfile.gettempdir()) / "cv5"  # OpenCV 5 for the DNN checks, kept apart from Kaggle's OpenCV 4
if REPO_FILES:
    HERE = Path(tempfile.gettempdir()) / "drishyam" / "training"
    for rel, text in REPO_FILES.items():
        (HERE.parent / rel).parent.mkdir(parents=True, exist_ok=True)
        (HERE.parent / rel).write_text(text)
else:
    HERE = Path("training").resolve() if NOTEBOOK else Path(__file__).resolve().parent
if KAGGLE and __name__ == "__main__":
    # Pinned: what this pipeline adds; torch, torchvision, timm and numpy stay Kaggle's own (pinning them would swap
    # the CUDA build). onnx is pinned too: Kaggle forces protobuf 5.29.5, which recent onnx wheels cannot load, and
    # this one pulls protobuf >= 6.31 (only unused TensorFlow-side packages mind). OpenCV 5 goes to its own folder:
    # Kaggle holds cv2 at 4.x, and two OpenCV wheels in one site-packages overwrite each other's cv2/. Training
    # only needs 4.x; the DNN checks (export and parity, run as child processes) need 5.
    pins = ["segmentation-models-pytorch==0.5.0", "onnx==1.23.2", "onnxslim==0.1.98", "onnxconverter-common==1.16.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins], check=True)
    if not (CV5 / "cv2").is_dir():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", str(CV5),
                        "opencv-python-headless==5.0.0.93"], check=True)
sys.path.insert(0, str(HERE))
from dacl_common import MEAN, STD, build_model, load_class_map

In [ ]:
# Data
def find_root(path: str) -> Path:
    """The folder holding annotations/ and images/ (an unzip or a Kaggle mount adds wrapper folders)."""
    p = Path(path)
    for depth in range(5):
        for c in sorted(p.glob("/".join(["*"] * depth))) if depth else [p]:
            if (c / "annotations" / "train").is_dir():
                return c
    raise FileNotFoundError(f"no annotations/train under {path}: on Kaggle, Add Input > winterqi131/dacl10k-v2-devphase;"
                            " otherwise see training/README.md for the layout")


class Dacl10k(Dataset):
    """Official split: annotations/{train,validation}/*.json (labelme polygons) + images/<split>/*.jpg.

    Images are resized to size x size with INTER_AREA, exactly like OnnxSegmenter does at inference (and
    like the dacl10k paper's 512 x 512 baselines). Train samples are, 70% of the time, a random zoomed crop
    of the full-resolution photo first, which is what the virtual PTZ feeds the model. Polygons are
    rasterised straight at the target size, one channel per class (a pixel can carry several classes).
    """

    def __init__(self, root: Path, split: str, size: int, names: list[str], train: bool, seed: int):
        self.ann = sorted((root / "annotations" / split).glob("*.json"))
        if not self.ann:
            raise FileNotFoundError(f"no annotations in {root / 'annotations' / split}")
        self.img_dir = root / "images" / split
        self.size, self.train, self.seed, self.epoch = size, train, seed, 0
        self.index = {n: i for i, n in enumerate(names)}

    def __len__(self) -> int:
        return len(self.ann)

    def __getitem__(self, i: int):
        d = json.loads(self.ann[i].read_text())
        path = self.img_dir / d["imageName"]
        # EXIF orientation applied (cv2's default), as labelme applied it when the polygons were drawn: the 10 of 7,910
        # dacl10k photos with a 90-degree EXIF rotation are labelled in the rotated frame (their JSON size is transposed).
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        h, w = img.shape[:2]
        if (w, h) != (d["imageWidth"], d["imageHeight"]):
            raise ValueError(f"{path} decodes to {w}x{h} but {self.ann[i].name} says "
                             f"{d['imageWidth']}x{d['imageHeight']}: its polygons would be misplaced")
        S = self.size
        rng = np.random.default_rng((self.seed, self.epoch, i))  # same sample + epoch => same augmentation
        x0 = y0 = 0
        cw, ch = w, h
        if self.train and rng.random() < 0.7:
            side = rng.uniform(0.3, 1.0)
            cw, ch = max(S // 4, int(w * side)), max(S // 4, int(h * side))
            cw, ch = min(cw, w), min(ch, h)
            x0, y0 = int(rng.integers(0, w - cw + 1)), int(rng.integers(0, h - ch + 1))
        x = cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (S, S), interpolation=cv2.INTER_AREA)
        mask = np.zeros((len(self.index), S, S), np.uint8)
        scale = (S / cw, S / ch)
        for s in d["shapes"]:
            k = self.index.get(s["label"])
            if k is not None and len(s["points"]) >= 3:
                pts = (np.asarray(s["points"], np.float64) - (x0, y0)) * scale
                cv2.fillPoly(mask[k], [np.round(pts).astype(np.int32)], 1)
        if self.train:
            if rng.random() < 0.5:
                x, mask = x[:, ::-1], mask[:, :, ::-1]
            if rng.random() < 0.8:  # brightness / contrast
                x = np.clip(x.astype(np.float32) * rng.uniform(0.75, 1.25) + rng.uniform(-25, 25), 0, 255)
        rgb = cv2.cvtColor(np.ascontiguousarray(x).astype(np.uint8), cv2.COLOR_BGR2RGB)
        rgb = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
        # uint8 mask (a quarter of float32 through worker shared memory and pinning); the loops convert on the device
        return torch.from_numpy(rgb.transpose(2, 0, 1).copy()), torch.from_numpy(np.ascontiguousarray(mask))


class Synthetic(Dataset):
    """Fabricated noise images with blocky random masks. Smoke-testing only: it proves the pipeline runs."""

    def __init__(self, n: int, size: int, classes: int, seed: int):
        self.n, self.size, self.classes, self.seed, self.epoch = n, size, classes, seed, 0

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, i: int):
        g = torch.Generator().manual_seed(self.seed * 100003 + i)
        x = torch.randn(3, self.size, self.size, generator=g)
        m = (torch.rand(self.classes, self.size // 16, self.size // 16, generator=g) > 0.92).float()
        return x, nn.functional.interpolate(m[None], size=(self.size, self.size), mode="nearest")[0]

In [ ]:
# Metrics: IoU per class accumulated over the whole validation set, as in the dacl10k paper
def _counts(pred: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
    dims = (0, 2, 3)
    return torch.stack([(pred & gt).sum(dims), (pred & ~gt).sum(dims), (~pred & gt).sum(dims)])  # tp, fp, fn


def _iou(c: torch.Tensor) -> torch.Tensor:
    tp, fp, fn = c.double()
    den = tp + fp + fn
    return torch.where(den > 0, tp / den.clamp(min=1), torch.full_like(den, float("nan")))  # nan: class never seen


def _merge(t: torch.Tensor, groups) -> torch.Tensor:
    return torch.stack([t[:, g].any(1) for g in groups], 1)  # a Drishyam class is on if any mapped channel is


@torch.no_grad()
def evaluate(model, loader, device, groups, amp, max_batches=0):
    """(IoU for the 19 dacl10k classes, IoU for the 5 Drishyam defects after the class_map merge)."""
    model.eval()
    c19 = c5 = 0
    for b, (x, y) in enumerate(loader):
        if max_batches and b >= max_batches:
            break
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            logits = model(x.to(device))
        pred, gt = logits.float() > 0, y.to(device) > 0  # logit > 0 is sigmoid > 0.5
        c19 = c19 + _counts(pred, gt)
        c5 = c5 + _counts(_merge(pred, groups), _merge(gt, groups))
    return _iou(c19).cpu().numpy(), _iou(c5).cpu().numpy()

In [ ]:
# Training loop
def parse(argv=None):
    p = argparse.ArgumentParser(description="Train a dacl10k segmenter for Drishyam (see training/README.md)")
    p.add_argument("--data", help="dacl10k root (annotations/ + images/); required unless --synthetic")
    p.add_argument("--out", default="runs/dacl10k", help="checkpoints (last.pt, best.pt) and metrics.json")
    p.add_argument("--arch", choices=["unet", "deeplabv3plus"], default="unet")
    p.add_argument("--encoder", default="tu-mobilenetv3_large_100")
    p.add_argument("--encoder-weights", default="imagenet", help="'imagenet' (downloads) or 'none'")
    p.add_argument("--size", type=int, default=512, help="train/val resolution, a multiple of 32")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--weight-decay", type=float, default=1e-4)
    # macOS: DataLoader worker processes hang at interpreter exit here (torch 2.2.2 + Python 3.12.13, reproduced with
    # a 10-line script unrelated to this one), so default to in-process loading there.
    p.add_argument("--workers", type=int, default=0 if sys.platform == "darwin" else min(4, os.cpu_count() or 1))
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--max-steps", type=int, default=0, help="stop after N optimiser steps (smoke runs); 0 = off")
    p.add_argument("--resume", help="a last.pt to continue from, or 'auto': <out>/last.pt if there is one, else a "
                   "last.pt under /kaggle/input (an earlier notebook version's output added as input)")
    p.add_argument("--hours", type=float, default=0, help="start no epoch that would end after this many hours "
                   "(a Kaggle session ends at 12); 0 = off. Resume to continue")
    p.add_argument("--synthetic", action="store_true", help="fabricated tiny dataset, no downloads (smoke test)")
    p.add_argument("--log-every", type=int, default=50)
    a = p.parse_args(argv)
    if not a.synthetic and not a.data:
        p.error("--data is required unless --synthetic")
    if a.size % 32:
        p.error("--size must be a multiple of 32")
    return a


def run_info(start_epoch: int) -> dict:
    """This session's package versions and source, one entry of <out>/run_info.json (zipped with the weights). On
    Kaggle there is no git checkout: `source_sha256` (make_notebook.source_sha256 at the commit the notebook was
    built from) identifies the code instead."""
    from importlib import metadata

    def version(dist: str, path: list[str] | None = None) -> str | None:
        found = next(iter(metadata.distributions(name=dist, **({"path": path} if path else {}))), None)
        return found.version if found else None

    names = ("torch", "torchvision", "timm", "segmentation-models-pytorch", "onnx", "onnxslim", "onnxconverter-common",
             "numpy")
    try:
        git = subprocess.run(["git", "rev-parse", "HEAD"], cwd=HERE, capture_output=True, text=True, check=False).stdout.strip()
    except OSError:
        git = ""
    if not SOURCE_SHA256:
        from make_notebook import source_sha256
    return {"start_epoch": start_epoch, "python": sys.version.split()[0],
            "versions": {**{n: version(n) for n in names}, "cv2": cv2.__version__,
                         "cv2_export": version("opencv-python-headless", [str(CV5)]) if KAGGLE else cv2.__version__},
            "git_sha": git or None, "source_sha256": SOURCE_SHA256 or source_sha256()}


def auto_resume(out: Path, search: Path = Path("/kaggle/input")) -> str | None:
    """`--resume auto`: <out>/last.pt, else the newest last.pt under `search`, copied into <out> with the
    best.pt, metrics.json and run_info.json beside it so the history carries on."""
    if (out / "last.pt").exists():
        return str(out / "last.pt")
    # glob.glob, not Path.glob: pathlib's ** does not enter symlinked folders, and an input mount may be one
    found = sorted(map(Path, glob.glob(str(search / "**" / "last.pt"), recursive=True)), key=lambda f: f.stat().st_mtime)
    if not found:
        print(f"--resume auto: no last.pt in {out} or under {search}, starting from epoch 0")
        return None
    for name in ("last.pt", "best.pt", "metrics.json", "run_info.json"):
        if (found[-1].parent / name).exists():
            shutil.copy2(found[-1].parent / name, out / name)
    print(f"resuming from {found[-1]}")
    return str(out / "last.pt")


def main(argv=None):
    import segmentation_models_pytorch as smp

    a = parse(argv)
    t_run = time.time()
    random.seed(a.seed)
    np.random.seed(a.seed)
    torch.manual_seed(a.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    names, _, groups = load_class_map()
    C = len(names)

    if a.synthetic:
        a.encoder_weights, a.workers = "none", 0
        train_ds = Synthetic(16, a.size, C, a.seed)
        val_ds = Synthetic(8, a.size, C, a.seed + 1)
    else:
        root = find_root(a.data)
        train_ds = Dacl10k(root, "train", a.size, names, True, a.seed)
        val_ds = Dacl10k(root, "validation", a.size, names, False, a.seed)
    kw = {"num_workers": a.workers, "pin_memory": amp}
    train_dl = DataLoader(train_ds, a.batch_size, shuffle=True, drop_last=True,
                          generator=torch.Generator().manual_seed(a.seed), **kw)
    val_dl = DataLoader(val_ds, a.batch_size, shuffle=False, **kw)

    model = build_model(a.arch, a.encoder, None if a.encoder_weights == "none" else a.encoder_weights, C).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.weight_decay)
    total = a.max_steps or a.epochs * len(train_dl)
    warm = min(2 * len(train_dl), total // 10)  # the paper's baselines: cosine schedule, 2-epoch warm-up
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    # torch.amp.GradScaler is torch >= 2.3 (Kaggle); torch.cuda.amp's is deprecated there but is all 2.2.2 (the Mac) has
    scaler = (torch.amp.GradScaler("cuda", enabled=amp) if hasattr(torch.amp, "GradScaler")
              else torch.cuda.amp.GradScaler(enabled=amp))
    bce = nn.BCEWithLogitsLoss()
    dice = smp.losses.DiceLoss("multilabel", from_logits=True)

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    if a.resume == "auto":
        a.resume = auto_resume(out)
    start, step, best, history = 0, 0, -1.0, []
    if a.resume:
        ck = torch.load(a.resume, map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start, step, best = ck["epoch"] + 1, ck["step"], ck.get("best", -1.0)
        history = json.loads((out / "metrics.json").read_text()) if (out / "metrics.json").exists() else []
    # one entry per session: a run resumed on another Kaggle image says which epochs which versions trained
    info = json.loads((out / "run_info.json").read_text()) if a.resume and (out / "run_info.json").exists() else []
    (out / "run_info.json").write_text(json.dumps([*info, run_info(start)], indent=1))
    print(f"{a.arch}/{a.encoder}  device={device.type} amp={amp}  train={len(train_ds)} val={len(val_ds)}  "
          f"steps/epoch={len(train_dl)}  total_steps={total}  start epoch {start}")

    for epoch in range(start, a.epochs):
        train_ds.epoch = epoch
        model.train()
        t0, run = time.time(), 0.0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True).float()
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                logits = model(x)
            logits = logits.float()
            loss = bce(logits, y) + dice(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            run = loss.item() if run == 0.0 else 0.98 * run + 0.02 * loss.item()
            if step % a.log_every == 0 or step == 1:
                print(f"epoch {epoch} step {step}/{total} loss {run:.4f} lr {sched.get_last_lr()[0]:.2e}", flush=True)
            if a.max_steps and step >= a.max_steps:
                break

        iou19, iou5 = evaluate(model, val_dl, device, groups, amp, max_batches=2 if a.max_steps else 0)
        miou, miou5 = float(np.nanmean(iou19)), float(np.nanmean(iou5))
        print(f"epoch {epoch} done in {time.time() - t0:.0f}s  val mIoU(19) {miou:.4f}  Drishyam-5 mean {miou5:.4f}")
        for n, v in zip(names, iou19):
            print(f"  {n:<14}{v:.3f}")
        for n, v in zip(["crack", "spalling", "efflorescence", "exposed_rebar", "corrosion"], iou5):
            print(f"  [merged] {n:<14}{v:.3f}")
        history.append({"epoch": epoch, "step": step, "train_loss": run, "val_miou19": miou, "val_miou5": miou5,
                        "iou19": dict(zip(names, map(float, iou19))), "iou5": list(map(float, iou5))})
        # best.pt is what ships: chosen on the five defects the exported graph outputs, not on all 19 classes
        best = max(best, miou5)
        ck = {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
              "scaler": scaler.state_dict(), "epoch": epoch, "step": step, "best": best, "arch": a.arch,
              "encoder": a.encoder, "classes": C, "names": names, "val_miou19": miou, "val_miou5": miou5,
              "args": vars(a)}
        torch.save(ck, out / "last.pt.tmp")
        os.replace(out / "last.pt.tmp", out / "last.pt")  # every epoch; a killed session never leaves a torn file
        if miou5 >= best:
            torch.save(ck, out / "best.pt")
        (out / "metrics.json").write_text(json.dumps(history, indent=1))
        if a.max_steps and step >= a.max_steps:
            break
        if a.hours and epoch + 1 < a.epochs and time.time() - t_run + (time.time() - t0) > a.hours * 3600:
            print(f"--hours {a.hours}: stopping after epoch {epoch}, the next would end too late; resume to continue")
            break
    print(f"checkpoints in {out}  (best.pt: Drishyam-5 val mIoU {best:.4f})")


def tool(script: str, *args) -> int:
    """Run a training/ script in a child process and show its output; on Kaggle with OpenCV 5 first on the path."""
    env = dict(os.environ)
    if KAGGLE:
        env["PYTHONPATH"] = os.pathsep.join([str(CV5), *filter(None, [env.get("PYTHONPATH")])])
    r = subprocess.run([sys.executable, str(HERE / script), *map(str, args)], check=False, env=env,
                       capture_output=True, text=True)
    print(r.stdout + r.stderr)
    return r.returncode

In [ ]:
# Settings
# The notebook runs with these flags (or DACL_FLAGS, if set); a headless run takes them from its command line.
# Every flag is in parse() above. --hours 11 leaves time to export inside Kaggle's 12 h; --resume auto picks up
# a run that stopped (this session's /kaggle/working/run, or an earlier version's output added as input).
NOTEBOOK_FLAGS = "--data /kaggle/input --out /kaggle/working/run --resume auto --hours 11"
if __name__ == "__main__":  # true in a notebook too: importing this module (tests) skips every cell from here
    FLAGS = shlex.split(os.environ.get("DACL_FLAGS", NOTEBOOK_FLAGS)) if NOTEBOOK else sys.argv[1:]
    A = parse(FLAGS)
    OUT, ONNX = Path(A.out), Path(A.out) / "onnx"

In [ ]:
# Data and GPU check
if __name__ == "__main__":
    ROOT = None if A.synthetic else find_root(A.data)
    if ROOT:
        n_train, n_val = (len(list((ROOT / "annotations" / s).glob("*.json"))) for s in ("train", "validation"))
        full = (n_train, n_val) == (6935, 975)
        print(f"dacl10k at {ROOT}: {n_train} train, {n_val} validation"
              + ("" if full else "  (v2 devphase has 6935 and 975: is this the whole set?)"))
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name}, {props.total_memory / 2**30:.0f} GB  (torch {torch.__version__}, cv2 {cv2.__version__})")
    elif KAGGLE and not A.synthetic:
        raise SystemExit("no GPU: Settings > Accelerator > GPU T4 x2, then run again")
    else:
        print(f"no GPU: training on CPU, fine for --synthetic smoke runs only (torch {torch.__version__})")

In [ ]:
# Train: <out>/last.pt and metrics.json every epoch, best.pt at the best mean IoU of Drishyam's 5 defects
if __name__ == "__main__":
    main(FLAGS)

In [ ]:
# Export: FP32 and FP16 ONNX with the 19 -> 6 merge in the graph, loaded on OpenCV 5 DNN at 256 and 512
if __name__ == "__main__":
    CKPT = OUT / "best.pt" if (OUT / "best.pt").exists() else OUT / "last.pt"
    if tool("export_onnx.py", "--ckpt", CKPT, "--out", ONNX):
        raise SystemExit("export failed: see the output above")
    FP32, FP16 = ONNX / "drishyam_dacl10k.onnx", ONNX / "drishyam_dacl10k.fp16.onnx"

In [ ]:
# Parity: OnnxSegmenter on OpenCV 5 DNN vs PyTorch, 4 crops (validation images, or noise for --synthetic)
if __name__ == "__main__":
    samples = Path(tempfile.mkdtemp())
    val = sorted((ROOT / "images" / "validation").glob("*.jpg"))[:4] if ROOT else []
    noise = np.random.default_rng(0)
    for i in range(4):
        img = cv2.imread(str(val[i])) if i < len(val) else noise.integers(0, 256, (576, 768, 3), dtype=np.uint8)
        cv2.imwrite(str(samples / f"{i}.jpg"), cv2.resize(img, (768, 576), interpolation=cv2.INTER_AREA))
    PARITY = tool("parity.py", "--ckpt", CKPT, "--onnx", FP32, *(["--onnx-fp16", FP16] if FP16.exists() else []),
                  "--samples", samples, "--n", 4)
    PARITY_OK = PARITY in (0, 3)  # 3 is parity.FP16_OVER: FP32 within tolerance, only FP16 over (it is left out)

In [ ]:
# Package: <out>/drishyam_dacl10k.zip: best.onnx, best_fp16.onnx (if its parity passed), metrics.json, run_info.json
if __name__ == "__main__":
    ZIP = OUT / "drishyam_dacl10k.zip"
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for src, name in ((FP32, "best.onnx"), (FP16, "best_fp16.onnx"), (OUT / "metrics.json", "metrics.json"),
                          (OUT / "run_info.json", "run_info.json")):
            if src.exists() and (src != FP16 or PARITY == 0):
                z.write(src, name)
    print(f"{ZIP}: {', '.join(z.namelist())}, {ZIP.stat().st_size / 1e6:.1f} MB")
    if PARITY == 0:
        print("parity ok")
    elif PARITY_OK:
        print("FP32 parity ok, FP16 over its tolerance (table above): best_fp16.onnx is left out of the zip, so "
              "Drishyam's fast looks will use the FP32 net")
    else:
        print("PARITY FAILED (table above): do not install these weights")
    print("next, on your machine in the repo, with the zip in ~/Downloads (or give its path, or the folder Safari unpacked it into):")
    print("  scripts/install_model.sh ~/Downloads/drishyam_dacl10k.zip")
    if not NOTEBOOK and not PARITY_OK:
        sys.exit(1)